### LLM 답변 캐싱하기

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH04-Models")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH04-Models


In [2]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI

gpt = ChatOpenAI( # ChatOpenAI 객체를 생성
    temperature=0, # 생성되는 텍스트의 다양성을 조절하는 매개변수
    model_name="gpt-4o-mini", # 사용할 모델의 이름
)

answer = gpt.stream("사랑이 뭔가요??") # 스트리밍 출력
stream_response(answer) # 답변 출력

사랑은 복잡하고 다면적인 감정으로, 사람들 간의 깊은 유대감이나 애정을 의미합니다. 사랑은 여러 형태로 나타날 수 있으며, 가족 간의 사랑, 친구 간의 사랑, 연인 간의 사랑 등 다양한 관계에서 경험할 수 있습니다. 

사랑은 종종 기쁨, 행복, 안정감과 같은 긍정적인 감정을 동반하지만, 때로는 고통이나 상실감과 같은 부정적인 감정도 수반할 수 있습니다. 사랑은 서로를 이해하고 지지하며, 함께 성장하고 발전하는 과정에서 중요한 역할을 합니다. 

결국 사랑은 사람마다 다르게 느끼고 표현될 수 있는 감정이며, 각자의 경험에 따라 그 의미가 달라질 수 있습니다.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model_name="gpt-4o-mini") # 모델을 생성

prompt = PromptTemplate.from_template("{country}에 대해서 200 내외로 요약해줘") # 프롬프트 템플릿을 생성

chain = prompt | llm # 프롬프트와 모델을 연결하여 체인을 생성

In [4]:
%%time
response = chain.invoke({"country": "대한민국"}) # 체인을 실행
print(response.content) # 결과 출력

대한민국은 동아시아에 위치한 국가로, 한반도의 남쪽에 자리 잡고 있습니다. 북쪽은 북한과 경계를 두고 있으며, 동쪽은 일본과, 서쪽은 중국과 가깝습니다. 대한민국은 민주적 정치 체제를 갖추고 있으며, 서울이 수도입니다. 경제적으로는 세계에서 가장 빠르게 성장한 국가 중 하나로, IT, 자동차, 조선, 반도체 산업이 발달하였습니다. 문화적으로는 K-POP, 한류 드라마, 한국 음식 등이 세계적으로 인기를 끌고 있습니다. 교육 수준이 높고, 헬스케어 시스템이 잘 갖춰져 있는 사회입니다. 다양한 자연경관과 역사적인 유적지 또한 많은 관광객을 끌어옵니다.
CPU times: total: 31.2 ms
Wall time: 1.75 s


#### 인메모리 캐시

In [5]:
%%time
from langchain_core.globals import set_llm_cache # 캐시를 설정하기 위해 필요한 모듈을 가져옴
from langchain_core.caches import InMemoryCache # 인메모리 캐시를 가져옴

set_llm_cache(InMemoryCache()) # 인메모리 캐시를 설정

response = chain.invoke({"country": "한국"}) # 체인을 실행
print(response.content) # 결과 출력

한국은 동아시아에 위치한 한반도의 국가로, 공식적으로는 대한민국(한국)과 조선민주주의인민공화국(북한)으로 나뉘어 있습니다. 한국은 고대부터 이어진 풍부한 역사와 문화를 가지고 있으며, 한글이라는 독특한 문자 체계를 사용합니다. 경제적으로는 IT, 자동차, 조선업 등 다양한 산업에서 세계적인 경쟁력을 갖추고 있습니다. 한국의 대중문화, 특히 K-팝과 드라마는 전 세계적으로 큰 인기를 끌고 있으며, 이를 통해 한국의 문화가 글로벌하게 확산되고 있습니다. 자연 경관도 다양해, 산, 바다, 그리고 사계절의 변화가 뚜렷하여 관광 명소가 많이 있습니다. 한국은 민주주의를 기반으로 한 사회이며, 교육과 기술 발전에 대한 높은 열망을 가지고 있습니다.
CPU times: total: 15.6 ms
Wall time: 1.79 s


In [6]:
%%time
response = chain.invoke({"country": "한국"}) # 체인을 실행
print(response.content) # 결과 출력 (캐시된 결과를 가져옴-엄청 빠름)

한국은 동아시아에 위치한 한반도의 국가로, 공식적으로는 대한민국(한국)과 조선민주주의인민공화국(북한)으로 나뉘어 있습니다. 한국은 고대부터 이어진 풍부한 역사와 문화를 가지고 있으며, 한글이라는 독특한 문자 체계를 사용합니다. 경제적으로는 IT, 자동차, 조선업 등 다양한 산업에서 세계적인 경쟁력을 갖추고 있습니다. 한국의 대중문화, 특히 K-팝과 드라마는 전 세계적으로 큰 인기를 끌고 있으며, 이를 통해 한국의 문화가 글로벌하게 확산되고 있습니다. 자연 경관도 다양해, 산, 바다, 그리고 사계절의 변화가 뚜렷하여 관광 명소가 많이 있습니다. 한국은 민주주의를 기반으로 한 사회이며, 교육과 기술 발전에 대한 높은 열망을 가지고 있습니다.
CPU times: total: 0 ns
Wall time: 10.3 ms


### SQLite 캐싱

In [7]:
from langchain_community.cache import SQLiteCache # SQLite 캐시를 가져옴
from langchain_core.globals import set_llm_cache # 캐시를 설정하기 위해 필요한 모듈을 가져옴
import os

if not os.path.exists("cache"): # cache 디렉토리가 존재하지 않으면 생성하기
    os.makedirs("cache")

# SQLite 캐시를 사용
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db")) # SQLite 캐시를 설정

C:\Users\user\AppData\Local\Temp\ipykernel_22148\2460769096.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import SQLiteCache # SQLite 캐시를 가져옴


In [10]:
%%time
response = chain.invoke({"country": "한국"}) # 체인을 실행
print(response.content) # 결과 출력

한국, 공식적으로 대한민국은 동아시아에 위치한 국가로, 한반도의 남쪽 절반을 차지하고 있습니다. 서울이 수도이며, 문화, 경제, 기술의 중심지로 알려져 있습니다. 한국은 고유한 전통문화와 현대적인 삶이 어우러져 있으며, K-팝, 한식, 한국 드라마 등으로 세계적으로 인기를 얻고 있습니다. 또한, 20세기 중반 이후 빠른 경제 성장을 이룬 '한강의 기적'으로 자주 언급됩니다. 정치적으로는 민주주의 체제를 갖추고 있으며, 1950년대의 한국 전쟁 이후 북한과 분단되어 있습니다. 한국은 교육, IT 산업, 연구 개발에서 높은 수준을 자랑하며, 글로벌 무대에서도 활발한 역할을 하고 있습니다.
CPU times: total: 0 ns
Wall time: 0 ns


d:\psj0902\ex0917\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
